# Launch Morphology Mutation

Desktop multi-SWC morphology editor with drag-box selection.

Core controls in the window:
- `r` drag-box: add to selection
- `t`/`y`: thin/thicken
- `g`: grow branch
- `d`: click-to-draw branch to a picked 3D point
- `b`: split selected edges
- `a`: reparent using last two selected
- `x`: detach selected
- `j`: add pre->post connection spec
- `s`: save mutation bundle (SWCs + manifest + validation + connection specs)
- `z`: undo


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys


def _find_notebook_dir() -> Path:
    cwd = Path.cwd()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([
            base,
            base / 'Phase 2' / 'apps' / 'VIP_Glia_Sim' / 'notebooks',
            base / 'apps' / 'VIP_Glia_Sim' / 'notebooks',
        ])
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue
        if (candidate / 'launcher_paths.py').exists():
            return candidate
    raise RuntimeError('Could not find launcher_paths.py. Open this notebook from Digifly Public or set the notebook working directory near the repo.')


NOTEBOOK_DIR = _find_notebook_dir()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from launcher_paths import (  # noqa: E402
    find_app_root,
    resolve_flow_run_dir,
    resolve_phase2_root,
    resolve_python_bin,
    resolve_swc_dir,
    validate_swc_dir,
)


WORK_ROOT = find_app_root()
PHASE2_ROOT = resolve_phase2_root(WORK_ROOT)
APP_PATH = WORK_ROOT / 'tools' / 'morphology_mutation_app.py'
PYTHON_BIN = resolve_python_bin()

NEURON_IDS = [10000, 10002, 10068, 10110, 11446, 11654, ]
SWC_DIR = resolve_swc_dir(PHASE2_ROOT, NEURON_IDS, app_root=WORK_ROOT)
validate_swc_dir(SWC_DIR, NEURON_IDS)
MUTATION_TAG = 'split'
OUTPUT_ROOT = WORK_ROOT / 'notebooks' / 'debug' / 'outputs'

# Mutation defaults
THIN_FACTOR = 0.8
THICK_FACTOR = 1.2
GROW_LENGTH_UM = 18.0
GROW_SEGMENTS = 4
GROW_RADIUS_SCALE = 0.85

# Inter-neuron connection-spec defaults
CONNECTION_CHEMICAL = 1
CONNECTION_GAP = 0
CONNECTION_GAP_MODE = 'none'  # none|non_rectifying|rectifying

# View defaults for the desktop app
RENDER_MODE = 'neuroglancer'          # 'tube', 'skeleton', or 'neuroglancer' (key 'w' toggles standard modes; key '1' toggles neuroglancer)
NEUROGLANCER_QUALITY = 'ultra'        # 'auto', 'balanced', 'high', or 'ultra' (max clarity; heavier render)
START_SOLO = True                     # recommended for neuroglancer volume mode; press '-' in the app to toggle all neurons
START_NEURON_ID = int(NEURON_IDS[0]) if NEURON_IDS else None
SKELETON_LINE_WIDTH = 6.0     # used when render_mode='skeleton' and by the slider
VISUAL_STYLE = 'classic'      # legacy flag kept for CLI compatibility; app uses classic only

# Optional flow-movie export inputs. None auto-picks the newest matching finished Phase 2 run.
FLOW_RUN_DIR = None
FLOW_RUN_DIR, FLOW_RUNS_ROOTS = resolve_flow_run_dir(
    PHASE2_ROOT,
    SWC_DIR,
    NEURON_IDS,
    app_root=WORK_ROOT,
    explicit=FLOW_RUN_DIR,
)
FLOW_FOCUS_PAIR = (10000, 10068)
FLOW_FPS = 30
FLOW_FRAME_STRIDE = 4
FLOW_SPEED_UM_PER_MS = 25.0
FLOW_PULSE_SIGMA_MS = 18.0
FLOW_SYN_DELAY_MS = None
FLOW_THRESHOLD_MV = 0.0
FLOW_MAX_MS = 0.0
FLOW_DURATION_SEC = 20.0

print('PYTHON_BIN =', PYTHON_BIN)
print('APP_PATH    =', APP_PATH)
print('SWC_DIR     =', SWC_DIR)
print('NEURON_IDS  =', NEURON_IDS)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RENDER_MODE =', RENDER_MODE)
print('NEUROGLANCER_QUALITY =', NEUROGLANCER_QUALITY)
print('START_SOLO =', START_SOLO, 'START_NEURON_ID =', START_NEURON_ID)
print('SKELETON_LINE_WIDTH =', SKELETON_LINE_WIDTH)
print('VISUAL_STYLE =', VISUAL_STYLE)
print('FLOW_RUNS_ROOTS =')
for _root in FLOW_RUNS_ROOTS:
    print('  -', _root)
print('FLOW_RUN_DIR =', FLOW_RUN_DIR)
print('FLOW_FOCUS_PAIR =', FLOW_FOCUS_PAIR)


In [ ]:
cmd = [
    str(PYTHON_BIN),
    str(APP_PATH),
    '--swc-dir', str(SWC_DIR),
    '--phase2-root', str(PHASE2_ROOT),
    '--neuron-ids', ','.join(str(int(x)) for x in NEURON_IDS),
    '--output-root', str(OUTPUT_ROOT),
    '--tag', str(MUTATION_TAG),
    '--thin-factor', str(THIN_FACTOR),
    '--thick-factor', str(THICK_FACTOR),
    '--grow-length-um', str(GROW_LENGTH_UM),
    '--grow-segments', str(GROW_SEGMENTS),
    '--grow-radius-scale', str(GROW_RADIUS_SCALE),
    '--connection-chemical', str(CONNECTION_CHEMICAL),
    '--connection-gap', str(CONNECTION_GAP),
    '--connection-gap-mode', str(CONNECTION_GAP_MODE),
    '--render-mode', str(RENDER_MODE),
    '--skeleton-line-width', str(SKELETON_LINE_WIDTH),
    '--visual-style', str(VISUAL_STYLE),
    '--neuroglancer-quality', str(NEUROGLANCER_QUALITY),
]
if START_SOLO:
    cmd.append('--start-solo')
if START_NEURON_ID is not None:
    cmd.extend(['--start-neuron-id', str(int(START_NEURON_ID))])
if FLOW_RUN_DIR is not None:
    cmd.extend(['--flow-run-dir', str(Path(FLOW_RUN_DIR).expanduser().resolve())])
if FLOW_FOCUS_PAIR is not None:
    cmd.extend(['--flow-focus-pair', ','.join(str(int(x)) for x in FLOW_FOCUS_PAIR)])
cmd.extend(['--flow-fps', str(FLOW_FPS)])
cmd.extend(['--flow-frame-stride', str(FLOW_FRAME_STRIDE)])
cmd.extend(['--flow-speed-um-per-ms', str(FLOW_SPEED_UM_PER_MS)])
cmd.extend(['--flow-pulse-sigma-ms', str(FLOW_PULSE_SIGMA_MS)])
if FLOW_SYN_DELAY_MS is not None:
    cmd.extend(['--flow-syn-delay-ms', str(FLOW_SYN_DELAY_MS)])
cmd.extend(['--flow-threshold-mv', str(FLOW_THRESHOLD_MV)])
cmd.extend(['--flow-max-ms', str(FLOW_MAX_MS)])
cmd.extend(['--flow-duration-sec', str(FLOW_DURATION_SEC)])
print(shlex.join(cmd))


In [ ]:
import time

env = os.environ.copy()
LOG_DIR = OUTPUT_ROOT / '_launcher_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LAUNCH_LOG = LOG_DIR / f"morphology_mutation_launcher_{int(time.time())}.log"

with open(LAUNCH_LOG, 'w', encoding='utf-8') as _lf:
    mutation_proc = subprocess.Popen(
        cmd,
        cwd=str(WORK_ROOT),
        env=env,
        stdout=_lf,
        stderr=subprocess.STDOUT,
    )

time.sleep(1.0)
rc = mutation_proc.poll()
if rc is None:
    print(f'Launched morphology mutation app (PID={mutation_proc.pid}).')
    print('Launcher log =', LAUNCH_LOG)
else:
    print(f'Launch failed early (exit={rc}). Log: {LAUNCH_LOG}')
    try:
        _txt = LAUNCH_LOG.read_text(encoding='utf-8')
        print('--- launcher log tail ---')
        print(_txt[-4000:])
    except Exception as _e:
        print('Could not read launcher log:', _e)


In [ ]:
# Optional: stop a still-running desktop process.
if 'mutation_proc' in globals() and mutation_proc.poll() is None:
    mutation_proc.terminate()
    print('Sent terminate to PID', mutation_proc.pid)
else:
    print('No running mutation process found in this kernel.')


## Find Latest Mutation Manifest And Wire Into Sim Notebooks

After a mutation bundle is saved from the app, this section finds the latest manifest and prepares notebook-side simulation inputs.


In [ ]:
from pathlib import Path
import json

cand_dirs = sorted(
    [p for p in OUTPUT_ROOT.glob('morphology_mutation_*') if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

LATEST_MUTATION_MANIFEST = None
for d in cand_dirs:
    m = d / 'morphology_mutation_manifest.json'
    if m.exists():
        LATEST_MUTATION_MANIFEST = m.resolve()
        break

if LATEST_MUTATION_MANIFEST is None:
    raise FileNotFoundError(f'No morphology mutation manifest found under: {OUTPUT_ROOT}')

print('LATEST_MUTATION_MANIFEST =', LATEST_MUTATION_MANIFEST)

m = json.loads(LATEST_MUTATION_MANIFEST.read_text(encoding="utf-8"))
print('neuron_ids =', m.get('neuron_ids'))
print('phase2_overlay_dir =', m.get('phase2_overlay_dir'))
print('n_ops =', len(m.get('operations') or []))
print('n_connections =', len(m.get('connections') or []))

print("\n# Paste into glia_simulation.ipynb controls:")
print("MUTATION_USE_MANIFEST = True")
print(f'MUTATION_MANIFEST_JSON = r"{LATEST_MUTATION_MANIFEST}"')
print("MUTATION_INCLUDE_CONNECTIONS = True")

print("\n# Paste into glia_force_ko_sweep_parallel.ipynb controls:")
print("SWEEP_MUTATION_USE_MANIFEST = True")
print(f'SWEEP_MUTATION_MANIFEST_JSON = r"{LATEST_MUTATION_MANIFEST}"')
print("SWEEP_MUTATION_INCLUDE_CONNECTIONS = True")
